In [ ]:
'''
downloading the dataset and preprocessing
'''

In [2]:
# installing the dataset

import kagglehub

# Download latest version
path = kagglehub.dataset_download("datamunge/virusmnist")

print("Path to dataset files:", 'Kathan 2')

100%|██████████| 156M/156M [00:12<00:00, 13.2MB/s] 

Extracting files...


Path to dataset files: Kathan 2


In [1]:
# getting the count of samples in the folders

import os

def count_images_in_folder(folder_path):
    print(f"\nDataset: {folder_path}")
    print("-" * 40)
    
    total_images = 0
    
    # Loop through each class folder
    for class_name in sorted(os.listdir(folder_path)):
        class_path = os.path.join(folder_path, class_name)
        
        if os.path.isdir(class_path):
            num_images = len([
                f for f in os.listdir(class_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))
            ])
            
            total_images += num_images
            print(f"{class_name}: {num_images}")
    
    print("-" * 40)
    print(f"Total images: {total_images}")

# Set base path
base_path = "."

# Count for train and test
count_images_in_folder(os.path.join(base_path, "train"))
count_images_in_folder(os.path.join(base_path, "test"))
count_images_in_folder(os.path.join(base_path, "val"))



Dataset: ./train
----------------------------------------
0: 2107
1: 6470
2: 2549
3: 2006
4: 665
5: 5586
6: 12937
7: 6303
8: 2159
9: 2803
----------------------------------------
Total images: 43585

Dataset: ./test
----------------------------------------
0: 175
1: 496
2: 205
3: 176
4: 58
5: 456
6: 1003
7: 491
8: 173
9: 225
----------------------------------------
Total images: 3458

Dataset: ./val
----------------------------------------
0: 234
1: 718
2: 283
3: 222
4: 73
5: 620
6: 1437
7: 700
8: 239
9: 311
----------------------------------------
Total images: 4837


In [4]:
import os
from PIL import Image
import numpy as np

def analyze_image_dimensions(folder_path):
    widths = []
    heights = []
    
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                img_path = os.path.join(root, file)
                
                try:
                    with Image.open(img_path) as img:
                        w, h = img.size
                        widths.append(w)
                        heights.append(h)
                except:
                    print(f"Error reading: {img_path}")
    
    widths = np.array(widths)
    heights = np.array(heights)
    
    print(f"\nAnalysis for: {folder_path}")
    print("-" * 40)
    print(f"Total Images: {len(widths)}")
    print(f"Average Width:  {widths.mean():.2f}")
    print(f"Average Height: {heights.mean():.2f}")
    print(f"Min Size:  ({widths.min()}, {heights.min()})")
    print(f"Max Size:  ({widths.max()}, {heights.max()})")

# Run for train and test
analyze_image_dimensions("train")
analyze_image_dimensions("test")



Analysis for: train
----------------------------------------
Total Images: 48422
Average Width:  32.00
Average Height: 32.00
Min Size:  (32, 32)
Max Size:  (32, 32)

Analysis for: test
----------------------------------------
Total Images: 3458
Average Width:  32.00
Average Height: 32.00
Min Size:  (32, 32)
Max Size:  (32, 32)


In [2]:
# using GANs for balance the dataset

In [5]:
import os
import shutil
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import save_image
from tqdm import tqdm
from PIL import Image

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

TRAIN_PATH    = './train'
BALANCED_PATH = './train_balanced'   # output: balanced dataset
IMG_SIZE      = 32
LATENT_DIM    = 128
TARGET        = 3000                 # target samples per class
NUM_CLASSES   = 10

# classes that need GAN generation (below target)
MINORITY_CLASSES = {0: 2107, 3: 2006, 4: 665, 8: 2159, 2: 2549, 9: 2803}
# classes that need undersampling (above target)
MAJORITY_CLASSES = {1: 6470, 5: 5586, 6: 12937, 7: 6303}

# ─────────────────────────────────────────────
# DCGAN ARCHITECTURE
# Conditional GAN — generates images conditioned on class label
# so we can specifically generate samples for minority classes
# ─────────────────────────────────────────────

class Generator(nn.Module):
    """
    Input:  noise vector (LATENT_DIM) + class embedding
    Output: grayscale image (1, 32, 32)
    
    Architecture: 4x ConvTranspose2d blocks
    4x4 → 8x8 → 16x16 → 32x32
    """
    def __init__(self, latent_dim, num_classes, img_channels=1):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)

        self.init_size = 4   # starting spatial size before upsampling

        # project noise + label into 4x4 feature maps
        self.fc = nn.Linear(latent_dim + num_classes, 512 * self.init_size * self.init_size)

        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(512),

            # 4x4 → 8x8
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # 8x8 → 16x16
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # 16x16 → 32x32
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            # 32x32 → 32x32 (refine without changing size)
            nn.Conv2d(64, img_channels, 3, stride=1, padding=1),
            nn.Tanh()   # output in [-1, 1] matching Normalize((0.5,),(0.5,))
        )

    def forward(self, noise, labels):
        label_input = self.label_emb(labels)                    # (B, num_classes)
        gen_input   = torch.cat([noise, label_input], dim=1)    # (B, latent_dim + num_classes)
        out = self.fc(gen_input)
        out = out.view(out.size(0), 512, self.init_size, self.init_size)  # (B, 512, 4, 4)
        return self.conv_blocks(out)                            # (B, 1, 32, 32)


class Discriminator(nn.Module):
    """
    Input:  image (1, 32, 32) + class label (embedded and tiled)
    Output: real/fake scalar per sample

    Uses spectral norm for training stability (avoids mode collapse)
    """
    def __init__(self, num_classes, img_channels=1):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, IMG_SIZE * IMG_SIZE)

        # input channels = img_channels + 1 (tiled label map)
        self.model = nn.Sequential(
            # 32x32 → 16x16
            nn.utils.spectral_norm(nn.Conv2d(img_channels + 1, 64, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),

            # 16x16 → 8x8
            nn.utils.spectral_norm(nn.Conv2d(64, 128, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),

            # 8x8 → 4x4
            nn.utils.spectral_norm(nn.Conv2d(128, 256, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),

            # 4x4 → 1x1
            nn.utils.spectral_norm(nn.Conv2d(256, 1, 4, stride=1, padding=0)),
        )

    def forward(self, img, labels):
        # tile label embedding as extra channel
        label_map = self.label_emb(labels).view(labels.size(0), 1, IMG_SIZE, IMG_SIZE)
        d_input = torch.cat([img, label_map], dim=1)    # (B, 2, 32, 32)
        return self.model(d_input).view(-1)             # (B,)


# ─────────────────────────────────────────────
# GAN TRAINING
# ─────────────────────────────────────────────
def train_gan(train_path, num_epochs=100, lr=0.0002, beta1=0.5):
    """
    Trains a Conditional DCGAN on the full training set.
    Uses LSGAN loss (MSE instead of BCE) for more stable training.
    """
    transform = transforms.Compose([
        transforms.Grayscale(1),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    dataset    = ImageFolder(train_path, transform=transform)
    dataloader = DataLoader(dataset, batch_size=128, shuffle=True,
                            num_workers=4, pin_memory=True, drop_last=True)

    G = Generator(LATENT_DIM, NUM_CLASSES).to(device)
    D = Discriminator(NUM_CLASSES).to(device)

    # weight init (standard DCGAN practice)
    def weights_init(m):
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight, 0.0, 0.02)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.normal_(m.weight, 1.0, 0.02)
            nn.init.zeros_(m.bias)

    G.apply(weights_init)
    D.apply(weights_init)

    opt_G = optim.Adam(G.parameters(), lr=lr,       betas=(beta1, 0.999))
    opt_D = optim.Adam(D.parameters(), lr=lr * 0.5, betas=(beta1, 0.999))
    # D uses lower LR to avoid overpowering G too quickly

    criterion = nn.MSELoss()   # LSGAN: more stable than BCELoss

    print(f"\n🔧 Training Conditional GAN for {num_epochs} epochs...")

    for epoch in range(num_epochs):
        G.train(); D.train()
        g_losses, d_losses = [], []

        for real_imgs, labels in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            real_imgs = real_imgs.to(device)
            labels    = labels.to(device)
            batch     = real_imgs.size(0)

            real_targets = torch.ones(batch).to(device)
            fake_targets = torch.zeros(batch).to(device)

            # ── Train Discriminator ────────────────────────────────
            opt_D.zero_grad()

            # real loss
            d_real = D(real_imgs, labels)
            d_real_loss = criterion(d_real, real_targets)

            # fake loss
            noise      = torch.randn(batch, LATENT_DIM).to(device)
            fake_imgs  = G(noise, labels).detach()
            d_fake     = D(fake_imgs, labels)
            d_fake_loss = criterion(d_fake, fake_targets)

            d_loss = (d_real_loss + d_fake_loss) / 2
            d_loss.backward()
            opt_D.step()

            # ── Train Generator ────────────────────────────────────
            opt_G.zero_grad()

            noise     = torch.randn(batch, LATENT_DIM).to(device)
            fake_imgs = G(noise, labels)
            g_out     = D(fake_imgs, labels)
            g_loss    = criterion(g_out, real_targets)   # G wants D to say "real"

            g_loss.backward()
            opt_G.step()

            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())

        print(f"  Epoch {epoch+1:3d} | G Loss: {np.mean(g_losses):.4f} | D Loss: {np.mean(d_losses):.4f}")

    torch.save(G.state_dict(), "gan_generator.pth")
    print("✅ Generator saved to gan_generator.pth")
    return G


# ─────────────────────────────────────────────
# GENERATE IMAGES FOR MINORITY CLASSES
# ─────────────────────────────────────────────
def generate_minority_images(G, output_base_path):
    """
    Uses trained Generator to synthesize images for minority classes
    and saves them directly into class folders.
    """
    G.eval()
    print("\n🎨 Generating synthetic images for minority classes...")

    for class_idx, current_count in MINORITY_CLASSES.items():
        needed       = TARGET - current_count
        class_folder = os.path.join(output_base_path, str(class_idx))
        os.makedirs(class_folder, exist_ok=True)

        print(f"  Class {class_idx}: generating {needed} images...", end=" ")

        generated = 0
        batch_size = 64

        with torch.no_grad():
            while generated < needed:
                this_batch = min(batch_size, needed - generated)
                noise      = torch.randn(this_batch, LATENT_DIM).to(device)
                labels     = torch.full((this_batch,), class_idx, dtype=torch.long).to(device)
                imgs       = G(noise, labels)   # (B, 1, 32, 32) in [-1, 1]

                # de-normalize: [-1,1] → [0,1] → PIL → save as PNG
                imgs = (imgs + 1) / 2.0
                for i, img_tensor in enumerate(imgs):
                    img_pil  = transforms.ToPILImage()(img_tensor.cpu())
                    filename = os.path.join(class_folder, f"gen_{generated + i:06d}.png")
                    img_pil.save(filename)

                generated += this_batch

        print(f"✅ {needed} images saved")


# ─────────────────────────────────────────────
# COPY REAL IMAGES (minority classes keep all real images)
# ─────────────────────────────────────────────
def copy_real_images(src_path, dst_path, class_idx, max_count=None):
    """
    Copies real images from src to dst for a given class.
    If max_count is set, randomly samples that many (for undersampling).
    """
    src_class = os.path.join(src_path, str(class_idx))
    dst_class = os.path.join(dst_path, str(class_idx))
    os.makedirs(dst_class, exist_ok=True)

    all_files = [f for f in os.listdir(src_class)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]

    if max_count is not None and len(all_files) > max_count:
        all_files = random.sample(all_files, max_count)   # random undersample

    for fname in all_files:
        shutil.copy2(
            os.path.join(src_class, fname),
            os.path.join(dst_class, fname)
        )

    return len(all_files)


# ─────────────────────────────────────────────
# BUILD BALANCED DATASET
# ─────────────────────────────────────────────
def build_balanced_dataset(train_path, balanced_path, G):
    """
    Combines:
    - Real images for all classes (undersampled for majority)
    - GAN-generated images for minority classes
    Into a single balanced dataset at balanced_path
    """
    if os.path.exists(balanced_path):
        shutil.rmtree(balanced_path)
    os.makedirs(balanced_path)

    print(f"\n📁 Building balanced dataset at: {balanced_path}")
    print("-" * 50)

    # Step 1: Copy real images (undersample majority, keep all minority)
    for class_idx in range(NUM_CLASSES):
        if class_idx in MAJORITY_CLASSES:
            count = copy_real_images(train_path, balanced_path, class_idx, max_count=TARGET)
            print(f"  Class {class_idx}: kept {count} real images (undersampled from {MAJORITY_CLASSES[class_idx]})")
        else:
            count = copy_real_images(train_path, balanced_path, class_idx, max_count=None)
            print(f"  Class {class_idx}: copied {count} real images (all kept)")

    # Step 2: Generate synthetic images for minority classes
    generate_minority_images(G, balanced_path)

    # Step 3: Verify final counts
    print("\n📊 Final balanced dataset distribution:")
    print("-" * 40)
    total = 0
    for class_idx in range(NUM_CLASSES):
        class_folder = os.path.join(balanced_path, str(class_idx))
        count = len(os.listdir(class_folder))
        total += count
        bar = "█" * (count // 100)
        print(f"  Class {class_idx}: {count:5d}  {bar}")
    print(f"  Total: {total}")
    print("-" * 40)


# ─────────────────────────────────────────────
# VERIFY GENERATED IMAGE QUALITY  (optional but useful)
# ─────────────────────────────────────────────
def save_sample_grid(G, save_path="gan_samples.png", n_per_class=8):
    """
    Saves a grid of sample generated images (n_per_class per class)
    so you can visually inspect GAN quality before using them.
    """
    G.eval()
    all_imgs = []

    with torch.no_grad():
        for class_idx in range(NUM_CLASSES):
            noise  = torch.randn(n_per_class, LATENT_DIM).to(device)
            labels = torch.full((n_per_class,), class_idx, dtype=torch.long).to(device)
            imgs   = G(noise, labels)
            imgs   = (imgs + 1) / 2.0   # de-normalize
            all_imgs.append(imgs.cpu())

    grid = torch.cat(all_imgs, dim=0)   # (NUM_CLASSES * n_per_class, 1, 32, 32)
    save_image(grid, save_path, nrow=n_per_class, normalize=False)
    print(f"\n🖼️  Sample grid saved to {save_path} — inspect before training!")


# ─────────────────────────────────────────────
# MAIN — run everything
# ─────────────────────────────────────────────
if __name__ == "__main__":

    # Step 1: Train GAN
    # Increase epochs if quality is poor — 100 is a good starting point
    G = train_gan(TRAIN_PATH, num_epochs=100)

    # Step 2: Visually verify quality before using generated images
    save_sample_grid(G, save_path="gan_samples.png")
    print("\n⚠️  Please inspect 'gan_samples.png' before proceeding.")
    print("   If images look like noise, increase num_epochs in train_gan().")
    input("   Press ENTER to continue building balanced dataset...\n")

    # Step 3: Build the balanced dataset
    build_balanced_dataset(TRAIN_PATH, BALANCED_PATH, G)

    print("\n✅ Done! Use './train_balanced' as your new TRAIN_PATH in the model.")
    print("   Update your model training script:")
    print("   TRAIN_PATH = './train_balanced'")

Using device: cuda

🔧 Training Conditional GAN for 100 epochs...


  Epoch   1 | G Loss: 0.6817 | D Loss: 0.1776


  Epoch   2 | G Loss: 0.7280 | D Loss: 0.1625


  Epoch   3 | G Loss: 0.7155 | D Loss: 0.1691


  Epoch   4 | G Loss: 0.4766 | D Loss: 0.2131


...

  Epoch  98 | G Loss: 0.3931 | D Loss: 0.1895


  Epoch  99 | G Loss: 0.3942 | D Loss: 0.1894


  Epoch 100 | G Loss: 0.3950 | D Loss: 0.1893
✅ Generator saved to gan_generator.pth

🖼️  Sample grid saved to gan_samples.png — inspect before training!

⚠️  Please inspect 'gan_samples.png' before proceeding.
   If images look like noise, increase num_epochs in train_gan().


   Press ENTER to continue building balanced dataset...
 



📁 Building balanced dataset at: ./train_balanced
--------------------------------------------------
  Class 0: copied 2107 real images (all kept)
  Class 1: kept 3000 real images (undersampled from 6470)
  Class 2: copied 2549 real images (all kept)
  Class 3: copied 2006 real images (all kept)
  Class 4: copied 665 real images (all kept)
  Class 5: kept 3000 real images (undersampled from 5586)
  Class 6: kept 3000 real images (undersampled from 12937)
  Class 7: kept 3000 real images (undersampled from 6303)
  Class 8: copied 2159 real images (all kept)
  Class 9: copied 2803 real images (all kept)

🎨 Generating synthetic images for minority classes...
  Class 0: generating 893 images... ✅ 893 images saved
  Class 3: generating 994 images... ✅ 994 images saved
  Class 4: generating 2335 images... ✅ 2335 images saved
  Class 8: generating 841 images... ✅ 841 images saved
  Class 2: generating 451 images... ✅ 451 images saved
  Class 9: generating 197 images... ✅ 197 images saved

📊 F

In [2]:
# getting the count of samples in the folders

import os

def count_images_in_folder(folder_path):
    print(f"\nDataset: {folder_path}")
    print("-" * 40)
    
    total_images = 0
    
    # Loop through each class folder
    for class_name in sorted(os.listdir(folder_path)):
        class_path = os.path.join(folder_path, class_name)
        
        if os.path.isdir(class_path):
            num_images = len([
                f for f in os.listdir(class_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))
            ])
            
            total_images += num_images
            print(f"{class_name}: {num_images}")
    
    print("-" * 40)
    print(f"Total images: {total_images}")

# Set base path
base_path = "."

# Count for train and test
count_images_in_folder(os.path.join(base_path, "virus_MNIST dataset/train_balanced_v2"))
count_images_in_folder(os.path.join(base_path, "virus_MNIST dataset/test"))
count_images_in_folder(os.path.join(base_path, "virus_MNIST dataset/val"))



Dataset: ./virus_MNIST dataset/train_balanced_v2
----------------------------------------
0: 2949
1: 6470
2: 3568
3: 2808
4: 931
5: 5586
6: 12937
7: 6303
8: 3022
9: 3924
----------------------------------------
Total images: 48498

Dataset: ./virus_MNIST dataset/test
----------------------------------------
0: 175
1: 496
2: 205
3: 176
4: 58
5: 456
6: 1003
7: 491
8: 173
9: 225
----------------------------------------
Total images: 3458

Dataset: ./virus_MNIST dataset/val
----------------------------------------
0: 234
1: 718
2: 283
3: 222
4: 73
5: 620
6: 1437
7: 700
8: 239
9: 311
----------------------------------------
Total images: 4837
